### **Topic 1: Introduction to Structured Outputs**

**Explanation:**
Large Language Models (LLMs) typically return **unstructured text** responses, which are great for humans but difficult for other systems (like databases or APIs) to process programmatically. **Structured Output** refers to the practice of having an LLM return its response in a well-defined data format, such as JSON. This allows the output to be easily parsed, validated, and used by other machines and applications, enabling LLMs to integrate with other systems seamlessly.

**Use Cases:**
1.  **Data Extraction**: Extracting specific information (name, skills, marks) from resumes to store in a database.
2.  **API Building**: Processing customer reviews to extract topics, pros, cons, and sentiment, then serving this structured data through an API.
3.  **Building Agents**: Enabling an agent to extract precise parameters (like numbers for a calculation) from a user's text prompt to call a tool (like a calculator).

### **Topic 2: Two Approaches to Structured Output in LangChain**

**Explanation:**
LangChain provides two main ways to get structured output from LLMs, depending on the model's capabilities:

1.  **`.with_structured_output()` Method**: This is the preferred and simplest method for models that natively support structured output (like recent OpenAI models). You define the desired schema, attach it to your model using this method, and the model directly returns data in that structure.
2.  **Output Parsers**: For models that do not natively support structured output (e.g., many open-source models), you use Output Parsers. The model generates text, and a parser class then extracts and formats that text into your desired structure. This will be covered in the next video.

This video focuses on the **`.with_structured_output()`** method.

### **Topic 3: Defining the Output Schema (Three Methods)**

**Explanation:**
To tell the model what structure you want, you need to define a schema. LangChain's `.with_structured_output()` method accepts a schema defined in three different ways. The choice depends on your need for validation and cross-language compatibility.

| Feature | **TypedDict** | **Pydantic** | **JSON Schema** |
| :--- | :--- | :--- | :--- |
| **Purpose** | Type hints & basic structure | Data validation, default values, type coercion | Universal, cross-language compatibility |
| **Validation** | No | Yes (e.g., ranges, email format) | No |
| **Type Coercion** | No | Yes (e.g., string "32" to int 32) | No |
| **Output Type** | Python Dictionary | Pydantic Object | Python Dictionary |
| **Best For** | Simple, pure-Python projects | Python projects needing robust data handling | Multi-language projects (e.g., Python backend + JS frontend) |

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os


load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", api_key=api_key)

### **Topic 4: Method 1 - Using TypedDict** (Not recommended, use JSON or pydantic)

**Explanation:**
`TypedDict` is a way to create a dictionary in Python where you specify the expected keys and their value types. It acts as a blueprint, providing type hints for better code clarity but does not enforce validation at runtime.

In [2]:
# schema
from typing import TypedDict, Annotated, Optional, Literal

class Review(TypedDict):
    key_themes: Annotated[
        list[str], "write down all the themes discussed in the review in a list"
    ]
    summary: Annotated[str,"A brief summary of the review"]
    sentiment: Annotated[
        Literal["pos", "neg"],
        "Return sentiment of the review either negative,positive or neutral",
    ]
    pros: Annotated[Optional[list[str]], "Write down all the pros inside a list"]
    cons: Annotated[Optional[list[str]], "Write down all the cons inside a list"]
    name: Annotated[Optional[str], "Write down the name of the reviewer"]




In [3]:
structured_model = model.with_structured_output(Review)

result = structured_model.invoke(
    """I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it’s an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything lightning fast—whether I’m gaming, multitasking, or editing photos. The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often. What really blew me away is the 200MP camera—the night mode is stunning, capturing crisp, vibrant images even in low light. Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it a bit uncomfortable for one-handed use. Also, Samsung’s One UI still comes with bloatware—why do I need five different Samsung apps for things Google already provides? The $1,300 price tag is also a hard pill to swallow.

Pros:
Insanely powerful processor (great for gaming and productivity)
Stunning 200MP camera with incredible zoom capabilities
Long battery life with fast charging
S-Pen support is unique and useful
                                 
Review by Kristal Shrestha """
)


In [4]:
print("Structured Output (Dictionary):")
print(result)
print(type(result))
print(f"\nAccessing Summary: {result['summary']}")
print(f"Accessing Sentiment: {result['sentiment']}")

Structured Output (Dictionary):
{'key_themes': ['performance', 'battery life', 'camera', 'S-Pen', 'design', 'software', 'price'], 'summary': "The Samsung Galaxy S24 Ultra is a powerful smartphone with an excellent camera, long battery life, and fast charging. While the S-Pen is a useful addition, the phone's size and weight, pre-installed bloatware, and high price point are notable drawbacks.", 'sentiment': 'pos', 'pros': ['Snapdragon 8 Gen 3 processor for fast performance', 'Long-lasting 5000mAh battery with 45W fast charging', 'Impressive 200MP camera with excellent night mode and usable zoom', 'Integrated S-Pen for note-taking and sketching', 'Stunning camera quality, especially in low light', '100x zoom capability, with good quality up to 30x'], 'cons': ['Heavy and large, making one-handed use uncomfortable', 'Comes with pre-installed bloatware (duplicate apps)', 'High price point of $1,300'], 'name': 'Samsung Galaxy S24 Ultra'}
<class 'dict'>

Accessing Summary: The Samsung Galaxy

In [5]:
result.keys()

dict_keys(['key_themes', 'summary', 'sentiment', 'pros', 'cons', 'name'])